In [1]:
import xarray

import settings
import thesis

In [2]:
# Use risk NetCDF files because although they are heavier than hazard files,
# they are fewer, hence fewer IO readings, and only the coordinates are read,
# hence no data is loaded in memory. Overall this is the optimal option.
dir_data = settings.dir_data / settings.site / "indicators" / "historical" / "reference" / "risk"
parameters_quantile = {}
for m in thesis.models:
    ds = xarray.open_dataset(dir_data / f"{m}.nc")
    for i, parameters in thesis.PARAMETERS_QUANTILE.items():
        dict_i = parameters_quantile.setdefault(i, {})  # Retrieve reference to the dict.
        for p in parameters:
            list_m = dict_i.setdefault(p, [])  # Retrieve reference to the list.
            k = f"{i}_{p}"
            da = ds[k]
            da = da.reset_coords(drop=True)
            values = thesis.PARAMETERS[i][p][:da.sizes[k]]  # Cheat: it is known that NaNs are at the end of the array.
            da = da.assign_coords(coords={k: (k, values)})
            da = da.rename({k: "quantile"})
            _ds = da.to_dataset(name=k)
            list_m.append(_ds)
    ds.close()

In [3]:
dim_m = "model"
dir_out = dir_data.parent / "parameters"
dir_out.mkdir(parents=True, exist_ok=True)
list_ds = []
for i, parameters in parameters_quantile.items():
    for p, list_m in parameters.items():
        ds = xarray.concat(list_m, dim=dim_m)
        ds = ds.assign_coords({dim_m: (dim_m, list(thesis.models))})
        k = f"{i}_{p}"
        ds.to_netcdf(dir_out / f"{k}.nc")
        
        # Evaluate statistics and display them.
        ds_stat = ds.count(dim=dim_m).rename({k: "sample_size"})
        ds_stat = ds_stat.assign({
            "mean": ds[k].mean(dim=dim_m, skipna=True),
            "std": ds[k].std(dim=dim_m, skipna=True, ddof=1)
        })
        print(k)
        display(ds_stat)
        
        # Save to CSV file.
        df = ds[k].to_dataset(dim="quantile").to_dataframe()
        header = [f"Order {q}" for q in thesis.PARAMETERS[i][p]]
        df.to_csv(
            path_or_buf=dir_out / f"{k}.csv",
            float_format=None,  # Round correctly using uncertainty.
            header=header,  # Custom header for the thesis document.
            index_label="Model"  # Custom header for the thesis document.
        )

heat_wave_index_thresh


<xarray.Dataset> Size: 140B
Dimensions:      (quantile: 5)
Coordinates:
  * quantile     (quantile) float64 40B 0.9 0.97 0.99 0.999 1.0
Data variables:
    sample_size  (quantile) int32 20B 22 22 22 22 22
    mean         (quantile) float64 40B 24.69 28.18 30.14 32.54 35.64
    std          (quantile) float64 40B 0.1426 0.1361 0.1733 0.5513 0.489

heat_wave_max_length_thresh_tasmax


<xarray.Dataset> Size: 140B
Dimensions:      (quantile: 5)
Coordinates:
  * quantile     (quantile) float64 40B 0.9 0.97 0.99 0.999 1.0
Data variables:
    sample_size  (quantile) int32 20B 22 22 22 22 22
    mean         (quantile) float64 40B 24.69 28.18 30.14 32.54 35.64
    std          (quantile) float64 40B 0.1426 0.1361 0.1733 0.5513 0.489

heat_wave_max_length_thresh_tasmin


<xarray.Dataset> Size: 140B
Dimensions:      (quantile: 5)
Coordinates:
  * quantile     (quantile) float64 40B 0.9 0.97 0.99 0.999 1.0
Data variables:
    sample_size  (quantile) int32 20B 22 22 22 22 16
    mean         (quantile) float64 40B 15.08 17.75 19.21 21.27 23.51
    std          (quantile) float64 40B 0.2207 0.2981 0.3619 0.5489 0.7397

wetdays_thresh


<xarray.Dataset> Size: 140B
Dimensions:      (quantile: 5)
Coordinates:
  * quantile     (quantile) float64 40B 0.9 0.97 0.99 0.999 0.9999
Data variables:
    sample_size  (quantile) int32 20B 22 22 22 22 22
    mean         (quantile) float64 40B 10.99 23.94 42.11 85.19 120.6
    std          (quantile) float64 40B 0.2668 0.7506 3.185 9.029 13.21

cwd_thresh


<xarray.Dataset> Size: 140B
Dimensions:      (quantile: 5)
Coordinates:
  * quantile     (quantile) float64 40B 0.9 0.97 0.99 0.999 0.9999
Data variables:
    sample_size  (quantile) int32 20B 22 22 22 22 22
    mean         (quantile) float64 40B 10.99 23.94 42.11 85.19 120.6
    std          (quantile) float64 40B 0.2668 0.7506 3.185 9.029 13.21